In [ ]:

# Step 3: Build syllable-level Bopomofo vocabulary

from pathlib import Path
from collections import Counter
import json
import re

import pandas as pd

OUT_DIR = Path("../npy")
META_PATH = OUT_DIR / "meta.csv"
VOCAB_PATH = OUT_DIR / "vocab.json"

BLANK_TOKEN = "<blank>"


In [ ]:

# Load meta.csv produced by Step 2

meta = pd.read_csv(META_PATH)

print("rows:", len(meta))
display(meta[["sentence", "bopomofo"]].head())


In [ ]:

# Split each Bopomofo string into tone-marked syllable tokens.
#
# Example:
#   ㄨㄛ3ㄇㄣ5ㄉㄜ5
# becomes:
#   ['ㄨㄛ3', 'ㄇㄣ5', 'ㄉㄜ5']

def split_bopomofo_syllables(text):
    text = str(text)
    return re.findall(r"[^1-5]+[1-5]", text)

meta["bopomofo_tokens"] = meta["bopomofo"].apply(
    lambda x: " ".join(split_bopomofo_syllables(x))
)

cols = meta.columns.tolist()
if "bopomofo_tokens" in cols:
    cols.remove("bopomofo_tokens")
pos = cols.index("bopomofo")
cols.insert(pos + 1, "bopomofo_tokens")
meta = meta[cols]

display(meta[["sentence", "bopomofo", "bopomofo_tokens"]].head(20))


In [ ]:
# Check that syllable splitting reconstructs the Bopomofo strings.

def reconstruct(text):
    return "".join(split_bopomofo_syllables(text))

bopomofo_text = meta["bopomofo"].fillna("").astype(str)
bad_mask = bopomofo_text != bopomofo_text.apply(reconstruct)
print("rows with split/reconstruction mismatch:", int(bad_mask.sum()))


In [ ]:

# Build syllable-level vocabulary.
#
# Each token is one complete syllable including its tone.

token_counts = Counter()

for text in meta["bopomofo"]:
    token_counts.update(split_bopomofo_syllables(text))

syllables = sorted(token_counts.keys())

id_to_token = [BLANK_TOKEN] + syllables
token_to_id = {token: i for i, token in enumerate(id_to_token)}

print("vocabulary size:", len(id_to_token))
print("number of syllable tokens:", len(syllables))
print("blank id:", token_to_id[BLANK_TOKEN])

display(
    pd.DataFrame(
        token_counts.most_common(),
        columns=["token", "count"],
    )
)


In [ ]:

# Save final syllable-level vocab.json and updated meta.csv

vocab = {
    "blank_token": BLANK_TOKEN,
    "blank_id": token_to_id[BLANK_TOKEN],
    "id_to_token": id_to_token,
    "token_to_id": token_to_id,
}

with open(VOCAB_PATH, "w", encoding="utf-8") as f:
    json.dump(vocab, f, ensure_ascii=False, indent=2)

meta.to_csv(META_PATH, index=False)

print("saved vocab:", VOCAB_PATH)
print("updated meta:", META_PATH)


In [ ]:

# Encode / decode sanity check

sample_idx = 0

tokens = split_bopomofo_syllables(meta.loc[sample_idx, "bopomofo"])
encoded = [token_to_id[token] for token in tokens]
decoded_tokens = [id_to_token[i] for i in encoded]

print("sentence:")
print(meta.loc[sample_idx, "sentence"])

print("\nbopomofo:")
print(meta.loc[sample_idx, "bopomofo"])

print("\ntokens:")
print(tokens)

print("\nencoded:")
print(encoded)

print("\ndecoded tokens:")
print(decoded_tokens)

print("\nmatch:", tokens == decoded_tokens)
